# YOLOv12n × VisDrone — `sfod` Notebook

> **SFOD: Small Feature-enhanced Object Detector**
>
> Cải tiến chính:
> 1. **P2 Detection Head** (stride=4, 160×160) — bắt vật thể rất nhỏ (people, bicycle)
> 2. **GhostConv PAN neck** — giảm ~40% params neck, compensate overhead của P2 head
> 3. **4-head Detect**: P2(32ch) + P3(64ch) + P4(128ch) + P5(256ch)
>
> Paper: TPH-YOLOv5 (ICCVW 2021) · GhostNet (CVPR 2020) · BGF-YOLOv10 (2024)
>
> T4 GPU | ~3–4h (120 epochs)

## 1. Kiểm tra GPU & CUDA

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode==0 else 'No GPU')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

## 3. Cài đặt dependencies

In [ ]:
%pip install -q ultralytics>=8.3.0 omegaconf>=2.3.0 rich>=13.0.0 thop
import ultralytics; ultralytics.checks()
print(f'ultralytics: {ultralytics.__version__}')

## 4. Upload project code

Chọn **một trong 3 cách** — uncomment dòng tương ứng:

In [ ]:
import os, shutil, zipfile
PROJECT_DIR = '/content/yolov12n-visdrone'

# -- Cách 1: Upload file zip (đơn giản nhất) ---------------------------
# from google.colab import files
# up = files.upload()   # chọn yolov12n-visdrone.zip
# with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall('/content/')

# -- Cách 2: Copy từ Google Drive (khuyến nghị nếu chạy nhiều lần) ----
# shutil.copytree('/content/drive/MyDrive/yolov12n-visdrone-code',
#                 PROJECT_DIR, dirs_exist_ok=True)

# -- Cách 3: Git clone --------------------------------------------------
# !git clone https://github.com/your-username/yolov12n-visdrone {PROJECT_DIR}

print('OK' if os.path.exists(PROJECT_DIR) else '⚠ Project chưa được upload')
import os; os.chdir(PROJECT_DIR); print('cwd:', os.getcwd())

## 5. Load VisDrone dataset từ Google Drive

> Dataset đã convert sang YOLO format và zip thành `VisDrone.zip`.
> Upload file này lên Drive tại: `MyDrive/yolov12n-visdrone/datasets/VisDrone.zip`

Thứ tự ưu tiên: VisDrone/ có sẵn → VisDrone.zip → báo lỗi.

In [ ]:
import os, zipfile, time
PROJ     = '/content/yolov12n-visdrone'
LOCAL    = PROJ + '/datasets/VisDrone'
BASE     = '/content/drive/MyDrive/yolov12n-visdrone/datasets'
ZIP_PATH = BASE + '/VisDrone.zip'
DIR_PATH = BASE + '/VisDrone'

os.makedirs(PROJ + '/datasets', exist_ok=True)

if os.path.exists(LOCAL):
    print(f'Dataset đã có sẵn: {LOCAL} ✓')
elif os.path.exists(ZIP_PATH):
    print(f'Giải nén từ {ZIP_PATH} ...')
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(PROJ + '/datasets')
    print(f'Xong trong {int(time.time()-t0)}s ✓')
elif os.path.exists(DIR_PATH):
    os.symlink(DIR_PATH, LOCAL)
    print(f'Symlink: {DIR_PATH} → {LOCAL} ✓')
else:
    raise FileNotFoundError(
        f'Không tìm thấy VisDrone.zip tại: {ZIP_PATH}\n'
        'Upload VisDrone.zip lên Drive trước khi chạy.')

In [ ]:
!python scripts/check_dataset.py --dataset-root datasets/VisDrone

## 6. Cấu hình training

In [ ]:
IDEA        = 'sfod'
MODEL       = 'yolov12n'
EPOCHS      = 120     # SFOD cần thêm epochs để P2 head hội tụ
BATCH       = 16      # giảm xuống 8 nếu CUDA OOM
IMGSZ       = 640
DEVICE      = '0'
TEST_EVERY  = 10      # test split sau mỗi 10 epochs
PROJECT_OUT = 'runs'
EXP_NAME    = f'{MODEL}_{IDEA}'

print(f'Model     : {MODEL} / {IDEA}')
print(f'Epochs    : {EPOCHS}')
print(f'Batch     : {BATCH}')
print(f'ImgSz     : {IMGSZ}')
print(f'Device    : GPU {DEVICE}')
print(f'Output    : {PROJECT_OUT}/{EXP_NAME}')
print()
print('SFOD Architecture:')
print('  Backbone  : YOLOv12n (layers 0-8) — pretrained yolo12n.pt')
print('  FPN       : layers 9-14 (A2C2f, near-identical to baseline)')
print('  P2 branch : GhostConv 128→32ch (NEW, 160×160)')
print('  PAN neck  : GhostConv (replaces A2C2f → fewer params)')
print('  Detect    : 4 heads — P2(32) P3(64) P4(128) P5(256)')

## 7. Training

Cell này chạy lâu nhất (~3-4h trên T4).
Mỗi 10 epochs sẽ in full metrics report trên test split.

In [ ]:
import os; os.makedirs('logs', exist_ok=True)
!python train.py \\
    --model      {MODEL} \\
    --idea       {IDEA} \\
    --epochs     {EPOCHS} \\
    --batch      {BATCH} \\
    --imgsz      {IMGSZ} \\
    --device     {DEVICE} \\
    --project    {PROJECT_OUT} \\
    --name       {EXP_NAME} \\
    --test-every {TEST_EVERY} \\
    --log-file   logs/{EXP_NAME}.log

## 7b. Phân tích kiến trúc SFOD sau khi load

Kiểm tra số params thực tế, tỷ lệ weight transfer từ backbone.

In [ ]:
from ultralytics import YOLO
import torch

# Khởi tạo SFOD model để xem kiến trúc
sfod = YOLO('configs/yolov12n_sfod.yaml')
n_params = sum(p.numel() for p in sfod.model.parameters())
n_trainable = sum(p.numel() for p in sfod.model.parameters() if p.requires_grad)

print(f'Total params    : {n_params:,} ({n_params/1e6:.3f}M)')
print(f'Trainable params: {n_trainable:,}')

# So sánh với baseline
baseline = YOLO('yolo12n.pt')
n_base = sum(p.numel() for p in baseline.model.parameters())
print(f'Baseline params : {n_base:,} ({n_base/1e6:.3f}M)')
print(f'Delta           : {(n_params-n_base)/1e3:+.1f}K params')

# Đếm layers
print(f'\nSFOD layers: {len(sfod.model.model)}')
for i, layer in enumerate(sfod.model.model):
    n = sum(p.numel() for p in layer.parameters())
    print(f'  [{i:2d}] {type(layer).__name__:<20} {n:>8,} params')

## 8. Đánh giá full metrics sau training

3 phần: Accuracy · Efficiency · Efficiency Ratios

In [ ]:
import os
WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt'
if not os.path.exists(WEIGHTS):
    WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/last.pt'
    print(f'best.pt not found, dùng: {WEIGHTS}')
else:
    print(f'Weights: {WEIGHTS}')

In [ ]:
# Val split
!python val.py \\
    --weights        {WEIGHTS} \\
    --data           configs/visdrone.yaml \\
    --split          val \\
    --imgsz          {IMGSZ} \\
    --device         {DEVICE} \\
    --model-name     {MODEL} \\
    --idea           {IDEA} \\
    --save-csv       runs/metrics.csv \\
    --benchmark-runs 30

In [ ]:
# Test split
!python val.py \\
    --weights        {WEIGHTS} \\
    --data           configs/visdrone.yaml \\
    --split          test \\
    --imgsz          {IMGSZ} \\
    --device         {DEVICE} \\
    --model-name     {MODEL} \\
    --idea           {IDEA} \\
    --save-csv       runs/metrics.csv \\
    --benchmark-runs 30

## 8b. So sánh SFOD vs Baseline

In [ ]:
import pandas as pd, os
if os.path.exists('runs/metrics.csv'):
    df = pd.read_csv('runs/metrics.csv')
    cols = ['model','idea','split','acc_map50','acc_map50_95',
            'acc_precision','acc_recall',
            'eff_gflops','eff_params_m','eff_fps',
            'ratio_map50_per_gflop','ratio_map50_x_fps']
    c = [x for x in cols if x in df.columns]
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)
    print(df[c].to_string(index=False))
    # Delta vs baseline
    if 'acc_map50' in df.columns:
        base_row = df[df['idea']=='baseline']
        sfod_row = df[df['idea']=='sfod']
        if not base_row.empty and not sfod_row.empty:
            b = base_row['acc_map50'].iloc[-1]
            s = sfod_row['acc_map50'].iloc[-1]
            print(f'\nmAP50 delta: {s-b:+.4f} ({100*(s-b)/b:+.1f}%)')
else:
    print('metrics.csv not found — chạy Section 8 trước')

## 9. Visualize training curves

In [ ]:
from IPython.display import Image, display
import os
exp_dir = f'{PROJECT_OUT}/{EXP_NAME}'
for name in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    p = os.path.join(exp_dir, name)
    if os.path.exists(p):
        print(name); display(Image(p, width=820))

In [ ]:
import pandas as pd, os
rcsv = f'{PROJECT_OUT}/{EXP_NAME}/results.csv'
if os.path.exists(rcsv):
    df = pd.read_csv(rcsv); df.columns = df.columns.str.strip()
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        best = df.loc[df[col].idxmax()]
        print(f'Best epoch : {int(best.get("epoch",0))}')
        print(f'mAP50      : {best[col]:.4f}')
        print(f'mAP50-95   : {best.get("metrics/mAP50-95(B)",0):.4f}')
        print(f'Precision  : {best.get("metrics/precision(B)",0):.4f}')
        print(f'Recall     : {best.get("metrics/recall(B)",0):.4f}')
    else:
        print(df.tail(3).to_string())

## 10. Lưu kết quả lên Google Drive

In [ ]:
import shutil, os
DRIVE_OUT = f'/content/drive/MyDrive/yolov12n-visdrone/runs/{EXP_NAME}'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy experiment folder
src = f'{PROJECT_OUT}/{EXP_NAME}'
if os.path.exists(src):
    shutil.copytree(src, DRIVE_OUT, dirs_exist_ok=True)
    print(f'Experiment saved → {DRIVE_OUT} ✓')

# Backup metrics.csv
mcsv = 'runs/metrics.csv'
if os.path.exists(mcsv):
    shutil.copy(mcsv, '/content/drive/MyDrive/yolov12n-visdrone/metrics.csv')
    print('metrics.csv backed up ✓')

# Download best.pt (optional)
# from google.colab import files
# files.download(f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt')

## 11. Demo Inference (tùy chọn)

In [ ]:
import glob, random, os
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO(WEIGHTS)
imgs = glob.glob('datasets/VisDrone/images/test/*.jpg')
samples = random.sample(imgs, min(3, len(imgs)))

results = model.predict(
    source=samples, conf=0.25, imgsz=IMGSZ, device=DEVICE,
    save=True, project='runs/predict', name=f'demo_{IDEA}'
)

plt.figure(figsize=(18, 6))
for i, r in enumerate(results):
    img = r.plot()[:, :, ::-1]
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(samples[i].split('/')[-1], fontsize=9)
    plt.axis('off')
plt.suptitle('SFOD Predictions (P2+P3+P4+P5 heads)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()